In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from enum import Enum
from dataclasses import dataclass
import math
from tqdm import tqdm
import sys, inspect

from tiles import *
from utils import *
import solver as solver_import

In [ ]:
def plot_schedule(schedule):
    methods = inspect.getmembers(schedule, inspect.ismethod)
    methods = [(i,j) for i,j in methods if not i.startswith('_')]

    com_ax = plt.subplot()
    com_ax.set_title('Combined')

    fig, ax = plt.subplots(1, len(methods), figsize=(3.5*len(methods), 3))

    for i, (m_name, m_func) in enumerate(methods):
        # print(i, m_name,m_func)
        g = list(map(m_func, np.arange(0,schedule.max_iter))) 
        ax[i].plot(g)
        ax[i].set_title(f'{m_name} max:{max(g):.3f} min:{min(g):.3f}')

        com_ax.plot(g)
    plt.plot()

In [ ]:
class Schedule:
    def __init__(self, max_iter):
        self.max_iter = max_iter
    def lr(self,x):
        x /= self.max_iter
        return ( np.cos(x*2*np.pi/4)**32 * 5 + np.cos(x*2*np.pi/4)**2 + np.cos(x*2*np.pi*5 % (np.pi/2))/5 + np.cos(x*2*np.pi*2 % (np.pi/2))/2 ) /6.7

    def temp(self,x):
        x /= self.max_iter
        if x<0.5:
            return np.maximum( np.cos(x*2*np.pi/2)**2 + np.cos(x*2*np.pi*5 % (np.pi/2))/5 , 0.001) / 1.2
        else:
            return max(0.2*(1-(x-0.5)*4),0)
    def logits_penalty(self,x):
        x /= self.max_iter
        return np.cos(x*np.pi/2)**2
    
    def decision_penalty(self,x):
        x /= self.max_iter

        x /=2
        return np.cos(x*np.pi/2 + 1/2*np.pi)**4 *4
    def empty_pos_penalty(self,x):
        x /= self.max_iter

        x /=2
        return np.cos(x*np.pi/2 + 1/2*np.pi)**8 *4 / 0.24

schedule = Schedule(50)
plot_schedule(schedule)

In [ ]:
BOX_SIZE = 15
INPUT_POS = (0,3)
OUT_POS = (13, 12)
NEG_INF = -500.0

In [ ]:
import importlib
importlib.reload(solver_import) # Fast reaload

sim = solver_import.FactorioSim(BOX_SIZE, DEFAULT_TILE_DATA)

In [ ]:
torch.autograd.set_detect_anomaly(False, check_nan=False)

In [ ]:
logits = torch.full((BOX_SIZE,BOX_SIZE,len(Tile)), -500.0)
active_tiles = list(range(0,12+1))

logits[:,:,active_tiles] = 5 * torch.randn(BOX_SIZE,BOX_SIZE,len(active_tiles))
logits.requires_grad_()

add_tensor = torch.zeros(BOX_SIZE,BOX_SIZE)
add_tensor[INPUT_POS[0], INPUT_POS[1]] = 100

MAX_ITER = 51
START_TEMP = 100

opt = torch.optim.Adam([logits],lr=25)
for iter in range(MAX_ITER):
    for g in opt.param_groups:
        g['lr'] = 100 * schedule.lr(iter)

    logits_noise = logits + 50 * schedule.temp(iter) * torch.randn_like(logits)
    with torch.no_grad():
        logits_noise[:, :, 12::] = -25000
    solve_log = torch.nn.functional.log_softmax(logits_noise/50,dim=-1)

    # Solve: (-inf,0)
    # logits_noise: (0,inf)

    items_log = sim.eval(solve_log,add_tensor, max_iter=BOX_SIZE*1.8)

    path_loss_log = -( items_log[OUT_POS[0], OUT_POS[1]] - np.log(100) )

    entropy_pc = -( solve_log.exp() * solve_log ).mean(-1) * 20
    # enthalpy = ce(solve_log[:,:,[i.value for i in Tile.BELTS()]],torch.as_tensor(1/13)) * 0.5

    belt_penalty_pc = (solve_log[:,:,[i.value for i in Tile.BELTS()]]).exp().sum(-1) * 40 # Linear field

    beta = 0.2 * max(0.0, 1.0 - iter/MAX_ITER/0.6)
    resurrect_pc = -solve_log[..., [i.value for i in Tile.BELTS()]].mean(-1) * beta

    loss = (path_loss_log.expand_as(entropy_pc)
            + belt_penalty_pc # removes unwanted belts 
            + resurrect_pc # helps correct belts live
            - schedule.logits_penalty(iter)*entropy_pc) # allows model to think more

    if iter%10==0:
        plot_solve(solve_log)

        fig, ax = plt.subplots(2,4,figsize=(3*4, 3*2))

        vmin = min([i.min().detach().cpu() for i in [loss,entropy_pc,belt_penalty_pc,resurrect_pc]])
        vmax = max([i.max().detach().cpu() for i in [loss,entropy_pc,belt_penalty_pc,resurrect_pc]])

        sns.heatmap(loss.detach().cpu(),           ax=ax[0][0],vmin=vmin,vmax=vmax).set_title('Total loss')
        sns.heatmap(entropy_pc.detach().cpu(),     ax=ax[0][1],vmin=vmin,vmax=vmax).set_title('Entropy')
        sns.heatmap(belt_penalty_pc.detach().cpu(),ax=ax[0][2],vmin=vmin,vmax=vmax).set_title('Belt penalty')
        sns.heatmap(resurrect_pc.detach().cpu(),   ax=ax[0][3],vmin=vmin,vmax=vmax).set_title('Resurrect')

        total_grad = torch.autograd.grad(loss.mean(), logits, retain_graph=True, allow_unused=True)[0].sum(-1).detach().cpu()
        sim_grad = torch.autograd.grad(path_loss_log, logits, retain_graph=True, allow_unused=True)[0].sum(-1).detach().cpu()
        penalty_grad = torch.autograd.grad(belt_penalty_pc.mean(), logits, retain_graph=True, allow_unused=True)[0].sum(-1).detach().cpu()

        vmin = total_grad.min()
        vmax = total_grad.max()

        sns.heatmap(total_grad.detach().cpu(),   ax=ax[1][0],vmin=vmin,vmax=vmax).set_title('Total grad')
        sns.heatmap(sim_grad.detach().cpu(),     ax=ax[1][1],vmin=vmin,vmax=vmax).set_title('Grad by sim')
        sns.heatmap(penalty_grad.detach().cpu(), ax=ax[1][2],vmin=vmin,vmax=vmax).set_title('Grad by penalty')

        plt.show()
    

    loss = loss.mean()

    loss.backward()
    opt.step()

    print(f'{iter/MAX_ITER*100:.1f}% Loss comb:{loss.item():.3f}, Path loss:{path_loss_log:.3f}, Belt penalty:{belt_penalty_pc.mean():.2f}, Resurrect: {resurrect_pc.mean():.2f}, Entropy:{entropy_pc.mean():.2f}')

    opt.zero_grad()
    # sch.step()

plot_solve(solve_log)

In [ ]:
-solve_log[2,5,Tile.EMPTY.value]

In [ ]:
sim.eval(solve_log, iter_add=add_tensor, verbose=10,max_iter=20)

In [ ]:
render_solve(solve_log)